Objectives:

1. Define a business-driven classification label (high-revenue trips).

2. Perform feature engineering using Spark DataFrames.

3. Handle missing values and outliers.

4. Analyse class distribution and imbalance.

5. Prepare ML-ready feature vectors.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

data_path = "/content/drive/MyDrive/bigdata_ml_project/data/raw/nyc_taxi"
df = spark.read.parquet(data_path)

In [3]:
from pyspark.sql.functions import (
    hour, dayofweek, unix_timestamp, when, col
)

# Trip duration
df = df.withColumn(
    "trip_duration_minutes",
    (unix_timestamp("tpep_dropoff_datetime") -
     unix_timestamp("tpep_pickup_datetime")) / 60
)

# Time-based features
df = df.withColumn("pickup_hour", hour("tpep_pickup_datetime")) \
       .withColumn("pickup_dayofweek", dayofweek("tpep_pickup_datetime"))

# Remove invalid trips
df = df.filter(col("trip_duration_minutes") > 0)

In [4]:
quantile_75 = df.approxQuantile("total_amount", [0.75], 0.01)[0]

df = df.withColumn(
    "high_revenue_trip",
    when(col("total_amount") >= quantile_75, 1).otherwise(0)
)

In [5]:
df = df.select(
    "passenger_count",
    "trip_distance",
    "trip_duration_minutes",
    "pickup_hour",
    "pickup_dayofweek",
    "PULocationID",
    "DOLocationID",
    "high_revenue_trip",
    "tpep_pickup_datetime"
)

In [6]:
df = df.filter(col("passenger_count") > 0)

In [7]:
df = df.filter(col("trip_distance") > 0)

In [8]:
df.groupBy("high_revenue_trip").count().show()

+-----------------+--------+
|high_revenue_trip|   count|
+-----------------+--------+
|                1|17708807|
|                0|51321229|
+-----------------+--------+



In [11]:
df = df.select(
    "passenger_count",
    "trip_distance",
    "trip_duration_minutes",
    "pickup_hour",
    "pickup_dayofweek",
    "PULocationID",
    "DOLocationID",
    "high_revenue_trip",
    "tpep_pickup_datetime"
).describe().show()

+-------+-----------------+-----------------+---------------------+------------------+-----------------+------------------+------------------+-------------------+
|summary|  passenger_count|    trip_distance|trip_duration_minutes|       pickup_hour| pickup_dayofweek|      PULocationID|      DOLocationID|  high_revenue_trip|
+-------+-----------------+-----------------+---------------------+------------------+-----------------+------------------+------------------+-------------------+
|  count|         69030036|         69030036|             69030036|          69030036|         69030036|          69030036|          69030036|           69030036|
|   mean|1.326095034920741|3.573598079392232|    17.48222498488475|14.400963400917247|4.126854547779752|164.97434145912948|164.41681957112118|0.25653770483329896|
| stddev|0.767942435099463|76.24676617976816|    32.81795158449586| 5.685807280404659|1.954027327892781|63.195572904413034| 69.28333306256063| 0.4367220095153284|
|    min|             

In [12]:
df.stat.corr("trip_distance", "total_amount")

AttributeError: 'NoneType' object has no attribute 'stat'

In [13]:
feature_cols = [
    "passenger_count",
    "trip_distance",
    "trip_duration_minutes",
    "pickup_hour",
    "pickup_dayofweek",
    "PULocationID",
    "DOLocationID"
]

In [14]:
df_ml = df.select(feature_cols + ["high_revenue_trip"])
df_ml.cache()
df_ml.count()

AttributeError: 'NoneType' object has no attribute 'select'

In [15]:
df_ml.unpersist()

NameError: name 'df_ml' is not defined